# Data Cleaning and Normalization

This notebook covers the data preprocessing pipeline for the Indian Job Market Analytics project:
1. Loading multi-source raw job and internship listings
2. Deduplicating entries
3. Standardizing city names and geographical categorizations
4. Parsing raw salary text into numeric bounds (min, max, average LPA)
5. Parsing experience ranges
6. Normalizing multi-valued skill strings into a 1NF relational table
7. Exporting to SQLite

In [1]:
import pandas as pd
import numpy as np
import re
import sqlite3
from pathlib import Path

RAW_DATA_PATH = Path('../data/raw/combined_raw_jobs.csv')
PROCESSED_DIR = Path('../data/processed')
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

### 1. Load Raw Dataset

In [2]:
df_raw = pd.read_csv(RAW_DATA_PATH)
print(f"Raw records loaded: {len(df_raw):,}")
df_raw.head(3)

Raw records loaded: 3,357


### 2. Deduplication and Normalization Functions

In [3]:
def standardize_city(loc):
    if pd.isna(loc):
        return 'Unknown'
    loc = str(loc).lower()
    if any(x in loc for x in ['bengaluru', 'bangalore']): return 'Bangalore'
    if any(x in loc for x in ['mumbai', 'bombay', 'navi mumbai']): return 'Mumbai'
    if any(x in loc for x in ['delhi', 'new delhi', 'gurgaon', 'gurugram', 'noida', 'ghaziabad', 'faridabad']): return 'Delhi NCR'
    if any(x in loc for x in ['hyderabad', 'secunderabad']): return 'Hyderabad'
    if any(x in loc for x in ['chennai', 'madras']): return 'Chennai'
    if any(x in loc for x in ['kolkata', 'calcutta']): return 'Kolkata'
    if any(x in loc for x in ['pune', 'puna']): return 'Pune'
    if any(x in loc for x in ['work from home', 'remote', 'wfh']): return 'Remote'
    if 'ahmedabad' in loc: return 'Ahmedabad'
    if 'jaipur' in loc: return 'Jaipur'
    if 'kochi' in loc: return 'Kochi'
    if 'indore' in loc: return 'Indore'
    if 'chandigarh' in loc: return 'Chandigarh'
    if 'coimbatore' in loc: return 'Coimbatore'
    if 'lucknow' in loc: return 'Lucknow'
    if 'nagpur' in loc: return 'Nagpur'
    return 'Other'

def categorize_role(title):
    if pd.isna(title):
        return 'Other'
    t = str(title).lower()
    if any(x in t for x in ['ml', 'machine learning', 'deep learning', 'ai engineer']): return 'ML Engineer'
    if any(x in t for x in ['data scientist', 'data science']): return 'Data Scientist'
    if any(x in t for x in ['data analyst', 'analytics', 'business analyst']): return 'Data Analyst'
    return 'Other'

def parse_salary(salary_text, job_type):
    if pd.isna(salary_text) or str(salary_text).strip() in ['Not disclosed', 'Unpaid', '']:
        return np.nan, np.nan, np.nan
    s = str(salary_text).lower().replace(',', '')
    if 'unpaid' in s or 'not disclosed' in s:
        return np.nan, np.nan, np.nan
    m = re.search(r'([\d.]+)\s*-\s*([\d.]+)\s*(?:lacs?|lakhs?|lpa)', s)
    if m:
        cmin, cmax = float(m.group(1)), float(m.group(2))
        return cmin, cmax, (cmin + cmax) / 2
    m2 = re.search(r'([\d.]+)\s*(?:lacs?|lakhs?|lpa)', s)
    if m2:
        val = float(m2.group(1))
        return val, val, val
    m3 = re.search(r'₹?\s*(\d+)\s*-\s*₹?\s*(\d+)\s*(?:/month|/\s*month|per month)', s)
    if m3:
        cmin = float(m3.group(1)) * 12 / 100000
        cmax = float(m3.group(2)) * 12 / 100000
        return cmin, cmax, (cmin + cmax) / 2
    m4 = re.search(r'₹?\s*(\d+)\s*(?:/month|/\s*month|per month)', s)
    if m4:
        val = float(m4.group(1)) * 12 / 100000
        return val, val, val
    return np.nan, np.nan, np.nan

def parse_experience(exp_text):
    if pd.isna(exp_text):
        return np.nan, np.nan
    e = str(exp_text).lower()
    if 'fresher' in e:
        return 0.0, 0.0
    m = re.search(r'(\d+)\s*-\s*(\d+)', e)
    if m:
        return float(m.group(1)), float(m.group(2))
    m2 = re.search(r'(\d+)\+', e)
    if m2:
        return float(m2.group(1)), np.nan
    return np.nan, np.nan

### 3. Execute Transformation Pipeline

In [4]:
df = df_raw.drop_duplicates(subset=['title', 'company', 'location', 'salary_text']).copy()
df['job_id'] = range(1, len(df) + 1)
df['city'] = df['location'].apply(standardize_city)
df['role_category'] = df['title'].apply(categorize_role)

salaries = df.apply(lambda row: pd.Series(parse_salary(row['salary_text'], row['job_type'])), axis=1)
df[['salary_min', 'salary_max', 'salary_avg']] = salaries

exps = df['experience'].apply(lambda x: pd.Series(parse_experience(x)))
df[['experience_min', 'experience_max']] = exps

print(f"Deduplication: {len(df_raw):,} -> {len(df):,} clean records.")
print("Role Category counts:")
print(df['role_category'].value_counts())

Deduplication: 3,357 -> 2,842 clean records.
Role Category counts:
Data Scientist    1329
Other             1080
Data Analyst       294
ML Engineer        139
Name: role_category, dtype: int64


### 4. Normalize Skills Table (1NF)

In [5]:
records = []
acronyms = {'Sql': 'SQL', 'Nlp': 'NLP', 'Aws': 'AWS', 'Gcp': 'GCP', 'Ml': 'ML', 'Etl': 'ETL', 'Ai': 'AI', 'Mlops': 'MLOps'}
for _, row in df.iterrows():
    if pd.notna(row['skills']):
        for s in str(row['skills']).split(','):
            s_clean = s.strip()
            if not s_clean:
                continue
            s_title = acronyms.get(s_clean.title(), s_clean.title())
            if 1 < len(s_title) < 50:
                records.append({'job_id': row['job_id'], 'skill': s_title})

df_skills = pd.DataFrame(records)
print(f"Normalized skill instances: {len(df_skills):,}")
print(f"Unique skills tracked: {df_skills['skill'].nunique():,}")
df_skills.head(5)

Normalized skill instances: 16,396
Unique skills tracked: 2,099


### 5. Save Cleaned Tables and Export to SQLite

In [6]:
output_cols = ['job_id', 'title', 'role_category', 'company', 'location', 'city', 'salary_text', 'salary_min', 'salary_max', 'salary_avg', 'experience', 'experience_min', 'experience_max', 'posted_date', 'job_type', 'source', 'scrape_date']
df_clean = df[output_cols]

df_clean.to_csv(PROCESSED_DIR / 'jobs_cleaned.csv', index=False)
df_skills.to_csv(PROCESSED_DIR / 'job_skills.csv', index=False)

conn = sqlite3.connect(PROCESSED_DIR / 'jobs.db')
df_clean.to_sql('jobs', conn, if_exists='replace', index=False)
df_skills.to_sql('job_skills', conn, if_exists='replace', index=False)
conn.close()
print("Successfully exported jobs and job_skills tables to SQLite (jobs.db)")

Successfully exported jobs and job_skills tables to SQLite (jobs.db)
